# Tic-tac-toe minimax algorithm with limited search
We will demonstrate the minimax algorithm on a game of tic-tac-toe played on a 3x3 board.

In [ ]:
import numpy as np

The State class captures the current state of the game.
* **Attributes**
    * gameplan - the game board with dimensions 3x3 and values 0 to 2
    * player - the player whose turn it is now
    * current_player - player will be used to analyze the game and to track the possible new states being created. Players alternate, so a new state must be viewed from the opponent's perspective. The player whose turn it is in the given state while searching the state space.
    * depth - the depth of the analyzed state
    * max_depth - how many moves ahead we look at most. If depth = max_depth, the game is no longer analyzed further.

* **Methods**
    * terminal_test - the method returns information on whether the current state is final. If so, the method returns the winner.
    * utility - the method tries to evaluate the current state from the player's perspective. In the basic variant, it only distinguishes whether the player wins with this move, doesn't win, or the move doesn't lead to the end of the game.
    * possible_actions - the method returns a list of possible moves. For tic-tac-toe, these will be the coordinates of the board where a game piece can be placed.
    * expand - this method takes the current state and the definition of an action (the coordinates where the piece is placed) and creates a new state of the game.
    * minmax - the actual implementation of the minimax algorithm
    * next_current_player - returns the opponent of the current_player variable
    * next_player - returns the opponent of the player variable    

In [ ]:
class State:
    """Captures the state of a tic-tac-toe game.

    gameplan       - 3x3 numpy array
                   - 0 - empty cell
                   - 1 - X
                   - 2 - O
    player         - the player whose turn it is in the actual game
    current_player - the player making a move in the current search state
    depth          - number of moves played so far
    max_depth      - maximum search depth from the current position
    """

    generated = 0

    def __init__(self, gameplan, player, current_player=None, depth=0, max_depth=3):
        self.gameplan = gameplan
        self.player = player
        if current_player is None:
            self.current_player = player
        else:
            self.current_player = current_player
        self.depth = depth
        self.max_depth = max_depth
        State.generated += 1

    def terminal_test(self):
        """Check whether the game is over and return the winner.

        Returns:
            0 - game is not over
            1 - player 1 wins
            2 - player 2 wins
        """
        for i in range(3):
            if np.array_equal(self.gameplan[i], [1, 1, 1]):
                return 1
            if np.array_equal(self.gameplan[i], [2, 2, 2]):
                return 2
            if np.array_equal(self.gameplan[:, i], [1, 1, 1]):
                return 1
            if np.array_equal(self.gameplan[:, i], [2, 2, 2]):
                return 2

        if np.array_equal(self.gameplan.diagonal(), [1, 1, 1]):
            return 1
        if np.array_equal(self.gameplan.diagonal(), [2, 2, 2]):
            return 2
        if np.array_equal(np.fliplr(self.gameplan).diagonal(), [1, 1, 1]):
            return 1
        if np.array_equal(np.fliplr(self.gameplan).diagonal(), [2, 2, 2]):
            return 2

        return 0

    def utility(self, result):
        """Evaluate the state from the perspective of self.player.

        Returns:
             0  - draw
             1  - self.player wins
            -1  - opponent wins
        """
        if result == 0:
            return 0
        elif result == self.player:
            return 1
        else:
            return -1

    def possible_actions(self):
        """Return a list of possible actions (coordinates of empty cells)."""
        possible_actions = []
        for i in range(3):
            for j in range(3):
                if self.gameplan[i][j] == 0:
                    possible_actions.append((i, j))
        return possible_actions

    def expand(self, select_action):
        """Apply the given action and return the resulting new State.

        Switches current_player to the opponent and increments depth.
        Returns None if the action is invalid.
        """
        if select_action[0] not in range(3):
            return None
        if select_action[1] not in range(3):
            return None
        if self.gameplan[select_action[0], select_action[1]] != 0:
            return None

        new_array = np.copy(self.gameplan)
        new_array[select_action[0], select_action[1]] = self.current_player
        return State(new_array,
                     self.player,
                     self.next_current_player(),
                     self.depth + 1,
                     max_depth=self.max_depth)

    def minmax(self, strategy="max", search_depth=0):
        """Select the best action using the minimax algorithm with depth limiting.

        strategy     - 'max' to maximise utility, 'min' to minimise it
        search_depth - levels searched so far in the current minimax call
        Returns (utility_value, action).
        """
        # terminal state — no action available, return utility directly
        result = self.terminal_test()
        if result != 0:
            return self.utility(result), None

        if strategy == "max":
            selected_utilization_value = float('-inf')
            next_strategy = "min"
        else:
            selected_utilization_value = float('inf')
            next_strategy = "max"

        actions = self.possible_actions()
        selected_action = actions[0]

        for action in actions:
            expanded_state = self.expand(action)

            result = expanded_state.terminal_test()

            if result != 0:
                return expanded_state.utility(result), action
            else:
                if len(expanded_state.possible_actions()) == 0:
                    utilization = 0
                else:
                    # !!! DOTO
                    # add condition depth limit reached — return neutral evaluation
                    utilization, _ = expanded_state.minmax(next_strategy, search_depth + 1)

                if utilization > selected_utilization_value and strategy == "max":
                    selected_utilization_value = utilization
                    selected_action = action
                elif utilization < selected_utilization_value and strategy == "min":
                    selected_utilization_value = utilization
                    selected_action = action

        return selected_utilization_value, selected_action

    def next_current_player(self):
        """Return the opponent of current_player."""
        return 3 - self.current_player

    def next_player(self):
        """Return the opponent of player."""
        return 3 - self.player

# Task
- Add a limit on the maximum search depth to the algorithm.
- Try different depth search limits.
- Observe how the times and number of generated states change
- Do the game results change?

You need to implement the condition at the place marked # !!! todo

In [ ]:
state = State(gameplan=np.array([[0, 0, 0], [0, 0, 0], [0, 0, 0]]),
              player=1, max_depth=2)

while True:
    # check if the game is over
    game_result = state.terminal_test()
    if game_result != 0:
        print(f"Winner is {game_result} ")
        break

    # check for draw
    if len(state.possible_actions()) == 0:
        print("Drawn")
        break

    # select best action using minimax with depth limit
    print(f"=====================\nPlayer {state.player}")
    _, player_action = state.minmax("max")
    print(f"Select action: {player_action}")
    state = state.expand(player_action)
    print(state.gameplan)
    print(f"Generated states {State.generated}.")
    State.generated = 0

    # switch to the other player
    state.player = state.next_player()